In [2]:
print("hello world")

hello world


In [3]:
# Test-1

import yaml
from dotenv import load_dotenv
import os

import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Load config.yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("✅ Environment loaded successfully")
print("✅ LLM Provider:", config["llm"]["provider"])
print("✅ OpenAI Key exists:", "OPENAI_API_KEY" in os.environ)
print("✅ Google Key exists:", "GOOGLE_API_KEY" in os.environ)

✅ Environment loaded successfully
✅ LLM Provider: openai
✅ OpenAI Key exists: True
✅ Google Key exists: True


In [4]:
from sentence_transformers import CrossEncoder

In [ ]:
# Test-2

from utils.loader import load_and_chunk_docs
from utils.retriever_faiss import get_retriever
from utils.reranker_cross_encoder import CrossEncoderReranker
import yaml

config = yaml.safe_load(open("config.yaml"))

# 1) chunks + base retriever (provider-aware FAISS)
chunks = load_and_chunk_docs("./data/raw/insurance_docs")
base_retriever = get_retriever(config, chunks_if_needed=chunks)

# 2) Stage-1 retrieve a larger candidate set (initial_k)
initial_k = config["reranking"]["initial_k"]
candidates = base_retriever.invoke("key terms of insurance contract?")[:initial_k] 
#this is a list of Document objects, top initial_k results
print(f"Stage-1 candidates: {len(candidates)}")

# 3) Stage-2 rerank down to final_k
reranker = CrossEncoderReranker() #object of CrossEncoderReranker class, which will use a cross-encoder model
#to rerank the candidates retrieved in Stage-1
final_k = config["reranking"]["final_k"]
reranked = reranker.rerank("key terms of insurance contract?", candidates, top_k=final_k)
#reranking will be done on top of the candidates retrieved in Stage-1, and will return top final_k results
#reranking returns a list of tuples (Document, score), sorted by score in descending order and top_k results

print("\nTop reranked results:")
for i, (doc, score) in enumerate(reranked, 1):
    preview = doc.page_content[:120].replace("\n", " ")
    print(f"{i}. score={score:.4f} :: {preview}...")


# **If you see below error**:
# Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`    

# Just do -> pip install hf_xet

✅ Loaded 2 docs → 8 chunks
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai
Stage-1 candidates: 5

Top reranked results:
1. score=6.9571 :: Standard Policy Terms and Definitions  This document provides simplified definitions for key terms used in your insuranc...
2. score=-7.9670 :: Effective Date  The date and time when coverage under the policy officially begins.  Deductible  The specific amount of ...
3. score=-8.2863 :: Coverage Limit  The maximum amount the insurance company will pay for a covered loss, as stated in the policy schedule. ...
4. score=-8.7083 :: IV. Final Decision and Settlement  The adjuster will issue a final determination:  APPROVED: Your claim meets the covera...
5. score=-10.7571 :: Medical records/bills (for injury claims).  III. Review and Assessment  Status: PROCESSING A dedicated Claims Adjuster w...


In [7]:
# Test-3

from utils.loader import load_and_chunk_docs
from utils.two_stage_retriever import TwoStageRetriever
import yaml

config = yaml.safe_load(open("config.yaml"))

# Load chunks only once → retriever auto builds FAISS index if missing
chunks = load_and_chunk_docs("./data/raw/insurance_docs")

retriever = TwoStageRetriever(chunks_if_needed=chunks)

query = "What are key terms in an insurance contract?"
results = retriever.invoke(query)

print(f"✅ Final reranked docs: {len(results)}\n")

for i, doc in enumerate(results):
    print(f"--- Doc #{i+1} ---")
    print(doc.page_content[:200].replace("\n", " "), "\n")

✅ Loaded 2 docs → 8 chunks
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai
✅ Final reranked docs: 5

--- Doc #1 ---
Standard Policy Terms and Definitions  This document provides simplified definitions for key terms used in your insurance contract. For full legal definitions, refer to the Schedule and Declarations P 

--- Doc #2 ---
Coverage Limit  The maximum amount the insurance company will pay for a covered loss, as stated in the policy schedule.  Exclusion  A specific event, property, or type of loss that is not covered by t 

--- Doc #3 ---
Effective Date  The date and time when coverage under the policy officially begins.  Deductible  The specific amount of money you must pay out-of-pocket before the insurer begins to cover the remainde 

--- Doc #4 ---
IV. Final Decision and Settlement  The adjuster will issue a final determination:  APPROVED: Your claim meets the coverage criteria. A settlement offer will be issued, or payment will be made directly 

--- Doc 